# Introduzione alla Modellazione Semantica con dbtQuesto notebook accompagna la lezione sulla **modellazione semantica** usando dbt e il database AdventureWorks.## Obiettivi della lezione1. Esplorare i dati grezzi (i CSV originali)2. Comprendere il problema del **fanout** (raddoppio conteggi)3. Capire il problema dei campi non allineati (header vs detail)4. Vedere come dbt risolve questi problemi automaticamente

In [ ]:
# Importa le librerie necessarieimport duckdbimport pandas as pd# Connettiti al database DuckDBcon = duckdb.connect("adventureworks-dbt-course/adventureworks/data/adventureworks.duckdb")# Mostra le tabelle disponibiliprint("Tabelle disponibili nel database:")tables = con.execute("SHOW TABLES").fetchall()for t in tables:    print(f"  - {t[0]}")

## 1. Esplorazione dei Dati GrezziI dati sono memorizzati in tabelle derivate dai file CSV (seeds). Vediamo cosa contiene ciascuna tabella.

In [ ]:
# Visualizza la tabella CLIENTIcustomers = con.execute("SELECT * FROM customers").df()print("TABELLA CLIENTI:")print(customers.to_string(index=False))

In [ ]:
# Visualizza la tabella ORDINI# Status: 1=pending, 5=shipped, 6=cancelledorders = con.execute("SELECT * FROM orders").df()print("TABELLA ORDINI:")print("(status: 1=pending, 5=shipped, 6=cancelled)")print(orders.to_string(index=False))

In [ ]:
# Visualizza la tabella RIGHE ORDINE# Contiene i singoli prodotti per ogni ordine, con scontiorder_lines = con.execute("SELECT * FROM order_lines").df()print("TABELLA RIGHE ORDINE:")print("(nota: discount = percentuale di sconto applicata)")print(order_lines.to_string(index=False))

---## 2. Il Problema del FanoutIl **fanout** è uno dei problemi più comuni nelle query SQL su dati transazionali.**Cosa succede:** quando fai un JOIN tra tabelle e poi un aggregazione senza le dovute accortezze, i conteggi vengono raddoppiati.**Esempio:** se un ordine ha 3 prodotti, facendo COUNT(*) dopo il JOIN, il cliente viene contato 3 volte!

In [ ]:
# QUERY SBAGLIATA: senza DISTINCT# Questa query non usa COUNT(DISTINCT), quindi conta le righequery_sbagliata = """SELECT    c.customer_id,    c.first_name || " " || c.last_name AS customer_name,    COUNT(ol.order_id) AS order_countFROM customers cJOIN orders o ON c.customer_id = o.customer_idJOIN order_lines ol ON o.order_id = ol.order_idGROUP BY c.customer_id, c.first_name, c.last_name"""result_sbagliato = con.execute(query_sbagliata).df()print("QUERY SBAGLIATA (senza DISTINCT):")print("Conta le righe, non gli ordini unici!")print()print(result_sbagliato.to_string(index=False))

In [ ]:
# QUERY CORRETTA: usa DISTINCT# COUNT(DISTINCT ...) conta solo gli ordini uniciquery_corretta = """SELECT    c.customer_id,    c.first_name || " " || c.last_name AS customer_name,    COUNT(DISTINCT o.order_id) AS order_countFROM customers cJOIN orders o ON c.customer_id = o.customer_idJOIN order_lines ol ON o.order_id = ol.order_idGROUP BY c.customer_id, c.first_name, c.last_name"""result_corretto = con.execute(query_corretta).df()print("QUERY CORRETTA (con DISTINCT):")print()print(result_corretto.to_string(index=False))

In [ ]:
# CONFRONTO DIRETTOcomparison = result_sbagliato.merge(result_corretto, on="customer_id", suffixes=("_sbagliato", "_corretto"))comparison["differenza"] = comparison["order_count_sbagliato"] - comparison["order_count_corretto"]print("="*65)print("DIFFERENZA TRA LE DUE QUERY:")print("="*65)print(" customer_name  | sbagliato | corretto | differenza")print("-" * 60)for _, row in comparison.iterrows():    print(f"{row["customer_name"]:14} | {row["order_count_sbagliato"]:9} | {row["order_count_corretto"]:8} | {row["differenza"]:9}")print()print("La differenza corrisponde al numero di prodotti per ordine!")

---## 3. Problema 2: Campi Non AllineatiUn altro errore comune è usare il campo total dellintestazione ordine invece di calcolare il revenue dalle righe.**Perché è un problema:**- Nelle righe ordine (order_lines) abbiamo gli sconti (discount)- Il campo orders.total è un valore pre-calcolato che potrebbe non essere allineato- Calcolando da order_lines otteniamo un valore più preciso e riconciliabile

In [ ]:
# QUERY SBAGLIATA: usa il campo total dellheader# Non considera gli sconti!query_header = """SELECT    c.customer_id,    c.first_name || " " || c.last_name AS customer_name,    SUM(o.total) AS lifetime_valueFROM customers cJOIN orders o ON c.customer_id = o.customer_idWHERE o.status = 5  -- solo ordini speditiGROUP BY c.customer_id, c.first_name, c.last_name"""result_header = con.execute(query_header).df()print("QUERY SBAGLIATA (usa orders.total):")print("Non considera gli sconti applicati!")print()print(result_header.to_string(index=False))

In [ ]:
# QUERY CORRETTA: calcola da order_lines con sconti# line_total * (1 - discount) = valore effettivoquery_lines = """SELECT    c.customer_id,    c.first_name || " " || c.last_name AS customer_name,    SUM(ol.line_total * (1 - ol.discount)) AS lifetime_valueFROM customers cJOIN orders o ON c.customer_id = o.customer_idJOIN order_lines ol ON o.order_id = ol.order_idWHERE o.status = 5  -- solo ordini speditiGROUP BY c.customer_id, c.first_name, c.last_name"""result_lines = con.execute(query_lines).df()print("QUERY CORRETTA (calcola da order_lines con sconti):")print()print(result_lines.to_string(index=False))

In [ ]:
# CONFRONTOcomp = result_header.merge(result_lines, on="customer_id", suffixes=("_header", "_lines"))comp["differenza"] = comp["lifetime_value_header"] - comp["lifetime_value_lines"]print("="*70)print("DIFFERENZA (gli sconti fanno la differenza!):")print("="*70)print(" customer_name  | da header  | da righe   | differenza")print("-" * 65)for _, row in comp.iterrows():    print(f"{row["customer_name"]:14} | {row["lifetime_value_header"]:10.2f} | {row["lifetime_value_lines"]:10.2f} | {row["differenza"]:10.2f}")

---## 4. Come dbt Risolve Questi ProblemiOra vediamo come i modelli dbt risolvono questi problemi **in automatico**.I modelli dbt (staging e marts) applicano le regole corrette per noi.

In [ ]:
# Visualizza fct_customers# Nota: order_count usa già COUNT(DISTINCT ...)#       lifetime calcola già con gli scontiprint("fct_customers - Fact Table Clienti")print()fct_customers = con.execute("SELECT customer_id, first_name, last_name, city, order_count, lifetime_gross_value, lifetime_net_value FROM fct_customers").df()print(fct_customers.to_string(index=False))

In [ ]:
# Visualizza fct_products# Nota: orders_with_product usa COUNT(DISTINCT ...)#       revenue calcola già con gli scontiprint("fct_products - Fact Table Prodotti")print()fct_products = con.execute("SELECT product_id, product_name, category_name, orders_with_product, units_sold, gross_revenue, net_revenue FROM fct_products").df()print(fct_products.to_string(index=False))

In [ ]:
# Visualizza fct_orders# Nota: confronta order_header_total vs gross_revenueprint("fct_orders - Fact Table Ordini")print()fct_orders = con.execute("SELECT order_id, order_date, customer_name, order_header_total, gross_revenue, net_revenue FROM fct_orders").df()print(fct_orders.to_string(index=False))

### Cosa fanno i modelli dbt:**fct_customers:**- COUNT(DISTINCT order_id) → evita il fanout- Calcola da order_lines con sconti → valore preciso**fct_products:**- COUNT(DISTINCT order_id) → ordini unici per prodotto- gross_revenue = somma di line_total- net_revenue = somma di line_total * (1 - discount)**fct_orders:**- Confronta order_header_total vs gross_revenue- Mostra limpatto degli sconti ordine per ordine

---## 5. Confronto FinaleLa modellazione semantica con dbt garantisce che:1. I filtri corretti sono applicati (es. status = shipped)2. Le aggregazioni usano DISTINCT dove necessario3. I calcoli sono fatti dai dettagli, non dai campi aggregati4. La logica è centralizzata e riutilizzabile**Risultato:** query semplici che funzionano subito!

In [ ]:
# Esempio: query semplice come dovrebbe essereprint("Con dbt, per ottenere i clienti per valore:")print()print("SELECT first_name, last_name, lifetime_net_value")print("FROM fct_customers")print("ORDER BY lifetime_net_value DESC;")print()print("RISULTATO:")simple = con.execute("SELECT first_name, last_name, lifetime_net_value FROM fct_customers ORDER BY lifetime_net_value DESC").df()print(simple.to_string(index=False))

---## 6. Esercizi PropostiOra tocca a te! Prova a risolvere questi esercizi:### Esercizio 1: Prodotti per Categoria**Domanda:** Quanti prodotti unici sono stati venduti per categoria?Suggerimento: Guarda la tabella fct_products### Esercizio 2: Prodotto più scontato**Domanda:** Qual è il prodotto con il maggiore sconto medio?Suggerimento: Confronta gross_revenue vs net_revenue in fct_products### Esercizio 3: Città più revenue**Domanda:** Quale città ha generato più revenue?Suggerimento: Guarda fct_customers e il campo city

In [ ]:
# Chiudi connessionecon.close()print("Notebook completato!")